In [1]:
import torch
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Using Apple Metal Performance Shaders (MPS)")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
#     print("Using CUDA GPU")
# else:
#     device = torch.device("cpu")
#     print("Using CPU")h
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 4080


In [6]:
from PIL import Image
import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import (
    PreTrainedTokenizerFast,
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import json

# ============ 1. Create tokenizer ============
tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print("Vocabulary:", tokenizer.get_vocab())

# ============ 2. Create model - UNFREEZE encoder ============
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-224",
    "gpt2"
)

model.decoder.resize_token_embeddings(len(tokenizer))

model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.vocab_size = len(tokenizer)

model.decoder.config.bos_token_id = tokenizer.bos_token_id
model.decoder.config.eos_token_id = tokenizer.eos_token_id
model.decoder.config.pad_token_id = tokenizer.pad_token_id
model.decoder.config.vocab_size = len(tokenizer)

# Keep encoder UNFROZEN but use very low learning rate
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# ============ 3. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, processor, tokenizer):
        self.entries = entries
        self.processor = processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["image"]).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values[0]
        
        sequence_text = " ".join(e["sequence"]) + " " + self.tokenizer.eos_token
        labels = self.tokenizer(sequence_text, return_tensors="pt", add_special_tokens=False).input_ids[0]

        return {"pixel_values": pixel_values, "labels": labels}

class MazeCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        labels = [item["labels"] for item in batch]
        max_len = max(len(l) for l in labels)
        
        padded_labels = []
        for label in labels:
            padding_length = max_len - len(label)
            padded = torch.cat([label, torch.full((padding_length,), -100, dtype=label.dtype)])
            padded_labels.append(padded)
        
        labels = torch.stack(padded_labels)
        return {"pixel_values": pixel_values, "labels": labels}

# ============ 4. Load small dataset ============
entries = json.load(open("data/grid_sequences.json"))
train_entries = entries[:100]  # Use 100 examples
print(f"Training on {len(train_entries)} examples\n")

dataset = MazeDataset(train_entries, image_processor, tokenizer)
collator = MazeCollator(tokenizer)

# ============ 5. Training with VERY LOW learning rate and MORE epochs ============
args = Seq2SeqTrainingArguments(
    output_dir="maze-model-unfrozen",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch = 16
    num_train_epochs=300,  # Much more epochs!
    learning_rate=1e-5,  # VERY low learning rate for fine-tuning
    warmup_steps=100,
    logging_steps=50,
    save_steps=10000,
    fp16=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    predict_with_generate=True,
    weight_decay=0.01,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
)

print("=" * 60)
print("TRAINING - This will take a while!")
print("=" * 60)
print(f"Epochs: {args.num_train_epochs}")
print(f"Learning rate: {args.learning_rate} (very low for full model fine-tuning)")
print(f"This should reach loss < 0.05 if it's going to work")
print()

result = trainer.train()

print(f"\nFinal loss: {result.training_loss:.6f}")

# ============ 6. Test ============
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("\n" + "=" * 60)
print("TESTING")
print("=" * 60)

correct = 0
for i in range(min(15, len(train_entries))):
    img = Image.open(train_entries[i]["image"])
    pixel_values = image_processor(img, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=20,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id,
        )
    
    predicted = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    expected = " ".join(train_entries[i]["sequence"])
    match = predicted == expected
    if match:
        correct += 1
    
    print(f"\nMaze {i}:")
    print(f"  Expected:  '{expected}'")
    print(f"  Predicted: '{predicted}'")
    print(f"  Match: {'✓' if match else '✗'}")

print(f"\n{'=' * 60}")
print(f"Accuracy: {correct}/15 ({100*correct/15:.1f}%)")
print(f"Final Loss: {result.training_loss:.6f}")
print(f"{'=' * 60}\n")

if result.training_loss < 0.05 and correct >= 12:
    print("✓✓✓ SUCCESS! ViT+GPT2 works!")
    print("Scale to full 924 examples with these settings.")
elif result.training_loss < 0.1:
    print("⚠ Close! Loss is decent but accuracy not perfect.")
    print("Try: (1) More epochs, or (2) Larger dataset")
elif result.training_loss > 0.2:
    print("✗ Not converging well. ViT+GPT2 may be too complex for this task.")
    print("Consider: Smaller encoder (e.g., ResNet18 instead of ViT)")

model.save_pretrained("maze-model-unfrozen/final")
tokenizer.save_pretrained("maze-model-unfrozen/final")
print("Model saved!")

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Vocabulary: {'U': 5, 'R': 4, '<s>': 1, '</s>': 2, '<pad>': 0, '<unk>': 3}


Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['transformer.h.0.crossattention.c_attn.bias', 'transformer.h.0.crossattention.c_attn.weight', 'transformer.h.0.crossattention.c_proj.bias', 'transformer.h.0.crossattention.c_proj.weight', 'transformer.h.0.crossattention.q_attn.bias', 'transformer.h.0.crossattention.q_attn.weight', 'transformer.h.0.ln_cross_attn.bias', 'transformer.h.0.ln_cross_attn.weight', 'transformer.h.1.crossattention.c_attn.bias', 'transformer.h.1.crossattention.c_attn.weight', 'transformer.h.1.crossattention.c_proj.bias', 'transformer.h.1.crossattention.c_proj.weight', 'transformer.h.1.crossattention.q_attn.bias', 'transformer.h.1.crossattention.q_attn.weight', 'transformer.h.1.ln_cross_attn.bias', 'transformer.h.1.ln_cross_attn.weight', 'transformer.h.10.crossattention.c_attn.bias', 'transformer.h.10.crossattention.c_attn.weight', 'transformer.h.10.crossattention.c_proj.bias', 'transformer.h.10.cros

Trainable parameters: 200,603,136
Training on 100 examples

TRAINING - This will take a while!
Epochs: 300
Learning rate: 1e-05 (very low for full model fine-tuning)
This should reach loss < 0.05 if it's going to work



Step,Training Loss
50,1.126700
100,0.646300
150,0.595200
200,0.550100
250,0.524500
300,0.511200
350,0.498000
400,0.471600
450,0.467600
500,0.455600



Final loss: 0.344104

TESTING

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R R R R U U U U U'
  Match: ✗

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R R R U U U R U U'
  Match: ✗

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R R R U R U U U U'
  Match: ✗

Maze 3:
  Expected:  'R R R R R U U U R U U U'
  Predicted: 'R R R R U R U U U U'
  Match: ✗

Maze 4:
  Expected:  'R R R R R U U U U R U U'
  Predicted: 'R R R R U U U R U U'
  Match: ✗

Maze 5:
  Expected:  'R R R R R U U U U U R U'
  Predicted: 'R R R R U U U U R U'
  Match: ✗

Maze 6:
  Expected:  'R R R R R U U U U U U R'
  Predicted: 'R R R R U U U U U R'
  Match: ✗

Maze 7:
  Expected:  'R R R R U R R U U U U U'
  Predicted: 'R R R U R U R U U U'
  Match: ✗

Maze 8:
  Expected:  'R R R R U R U R U U U U'
  Predicted: 'R R R U U U U U U'
  Match: ✗

Maze 9:
  Expected:  'R R R R U R U U R U U U'
  Predicted: 'R R R U R U R U U U'
  Match: ✗

Maze 10:
  Expected:  'R R R R

In [11]:
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import (
    PreTrainedTokenizerFast,
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    AutoImageProcessor,
    GPT2Config,
    GPT2LMHeadModel,
)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import json
import torchvision.models as torch_models

# ============ 1. Create tokenizer ============
print("=" * 60)
print("CREATING TOKENIZER")
print("=" * 60)

tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print(f"Vocabulary: {tokenizer.get_vocab()}")
print(f"Vocab size: {len(tokenizer)}\n")

# ============ 2. Create Custom Encoder-Decoder Model ============
class ResNetGPT2Model(nn.Module):
    """Custom encoder-decoder with ResNet18 encoder and GPT2 decoder"""
    def __init__(self, vocab_size, hidden_size=768):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        
        # ResNet18 encoder
        resnet = torch_models.resnet18(pretrained=True)
        self.encoder_features = nn.Sequential(*list(resnet.children())[:-1])
        self.encoder_projection = nn.Linear(512, hidden_size)
        
        # GPT2 decoder with cross-attention
        decoder_config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=512,
            n_ctx=512,
            n_embd=hidden_size,
            n_layer=6,
            n_head=12,
            add_cross_attention=True,
        )
        self.decoder = GPT2LMHeadModel(decoder_config)
        
    def forward(self, pixel_values, labels=None):
        # Encode images
        batch_size = pixel_values.shape[0]
        features = self.encoder_features(pixel_values)  # [batch, 512, 1, 1]
        features = features.squeeze(-1).squeeze(-1)  # [batch, 512]
        encoder_hidden_states = self.encoder_projection(features)  # [batch, 768]
        encoder_hidden_states = encoder_hidden_states.unsqueeze(1)  # [batch, 1, 768]
        
        if labels is not None:
            # Training mode - shift labels for teacher forcing
            # Input: labels without last token, Target: labels without first token
            decoder_input_ids = labels[:, :-1].contiguous()
            lm_labels = labels[:, 1:].contiguous()
            
            # Forward through decoder
            outputs = self.decoder(
                input_ids=decoder_input_ids,
                encoder_hidden_states=encoder_hidden_states,
                labels=lm_labels,
                return_dict=True
            )
            return outputs
        else:
            # Inference mode handled separately
            return encoder_hidden_states
    
    def generate(self, pixel_values, max_length=20, eos_token_id=2, bos_token_id=1, pad_token_id=0):
        """Generate sequences autoregressively"""
        self.eval()
        batch_size = pixel_values.shape[0]
        device = pixel_values.device
        
        # Encode images
        features = self.encoder_features(pixel_values)
        features = features.squeeze(-1).squeeze(-1)
        encoder_hidden_states = self.encoder_projection(features).unsqueeze(1)
        
        # Start with BOS token
        generated = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
        
        for _ in range(max_length - 1):
            outputs = self.decoder(
                input_ids=generated,
                encoder_hidden_states=encoder_hidden_states,
                return_dict=True
            )
            
            # Get next token
            next_token_logits = outputs.logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)
            
            # Append to sequence
            generated = torch.cat([generated, next_token], dim=1)
            
            # Check if all sequences have generated EOS
            if (next_token == eos_token_id).all():
                break
        
        return generated

# ============ 3. Create model ============
print("=" * 60)
print("CREATING MODEL (ResNet18 + GPT2)")
print("=" * 60)

model = ResNetGPT2Model(vocab_size=len(tokenizer), hidden_size=768)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")
print(f"  - Encoder (ResNet18): {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  - Decoder (GPT2-small): {sum(p.numel() for p in decoder.parameters()):,}\n")

# ============ 4. Custom Trainer for our model ============
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

def train_model(model, train_loader, epochs, lr, device):
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.6f}")
        
        # Test every 25 epochs
        if (epoch + 1) % 25 == 0:
            test_model(model, train_loader, device, tokenizer, num_samples=5)
            model.train()
    
    return avg_loss

def test_model(model, loader, device, tokenizer, num_samples=10):
    model.eval()
    dataset = loader.dataset
    
    print("\n" + "="*60)
    print("Sample Predictions:")
    print("="*60)
    
    correct = 0
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        pixel_values = sample['pixel_values'].unsqueeze(0).to(device)
        
        # Get expected sequence
        img_path = dataset.entries[i]["image"]
        expected_seq = dataset.entries[i]["sequence"]
        expected = " ".join(expected_seq)
        
        with torch.no_grad():
            generated = model.generate(
                pixel_values,
                max_length=20,
                eos_token_id=tokenizer.eos_token_id,
                bos_token_id=tokenizer.bos_token_id
            )
        
        # Decode
        predicted = tokenizer.decode(generated[0], skip_special_tokens=True)
        match = predicted == expected
        if match:
            correct += 1
        
        print(f"\nMaze {i}:")
        print(f"  Expected:  '{expected}'")
        print(f"  Predicted: '{predicted}'")
        print(f"  Match: {'✓' if match else '✗'}")
    
    print(f"\nAccuracy: {correct}/{num_samples}\n")
    return correct

# ============ 5. Image processor ============
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============ 6. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, transform, tokenizer):
        self.entries = entries
        self.transform = transform
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["image"]).convert("RGB")
        pixel_values = self.transform(img)
        
        # Add EOS token
        sequence_text = " ".join(e["sequence"]) + " " + self.tokenizer.eos_token
        labels = self.tokenizer(sequence_text, return_tensors="pt", add_special_tokens=False).input_ids[0]

        return {"pixel_values": pixel_values, "labels": labels}

class MazeCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        labels = [item["labels"] for item in batch]
        max_len = max(len(l) for l in labels)
        
        padded_labels = []
        for label in labels:
            padding_length = max_len - len(label)
            padded = torch.cat([label, torch.full((padding_length,), -100, dtype=label.dtype)])
            padded_labels.append(padded)
        
        labels = torch.stack(padded_labels)
        return {"pixel_values": pixel_values, "labels": labels}

# ============ 7. Load data ============
entries = json.load(open("data/grid_sequences.json"))

print("=" * 60)
print("DATASET")
print("=" * 60)
print(f"Total examples: {len(entries)}")

# Start with 100 examples to verify it works
train_entries = entries[:100]
print(f"Training on: {len(train_entries)} examples")
print(f"(Will scale to full {len(entries)} if this works)\n")

dataset = MazeDataset(train_entries, image_transform, tokenizer)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda b: collator(b))

# ============ 8. Training ============
collator = MazeCollator(tokenizer)

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Training examples: {len(train_entries)}")
print(f"Batch size: 8")
print(f"Epochs: 100")
print(f"Learning rate: 5e-4")
print()

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

final_loss = train_model(model, train_loader, epochs=100, lr=5e-4, device=device)

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"Final loss: {final_loss:.6f}")
print()

# ============ 9. Final Testing ============
model.eval()

print("=" * 60)
print("FINAL TESTING")
print("=" * 60)

correct = test_model(model, train_loader, device, tokenizer, num_samples=15)

print(f"{'=' * 60}")
print(f"RESULTS")
print(f"{'=' * 60}")
print(f"Accuracy: {correct}/15 ({100*correct/15:.1f}%)")
print(f"Final Loss: {final_loss:.6f}")
print(f"{'=' * 60}\n")

# ============ 10. Evaluation and next steps ============
if final_loss < 0.05 and correct >= 12:
    print("🎉 SUCCESS! ResNet18 + GPT2 works!")
    print("✅ Loss converged below 0.05")
    print("✅ High accuracy on training examples")
    print("\n📈 NEXT STEPS:")
    print("1. Train on full 924 examples with same settings")
    print("2. Implement train/test split to evaluate generalization")
    print("3. Your vision-to-sequence model is working!")
elif final_loss < 0.15 and correct >= 8:
    print("⚠️  GOOD PROGRESS - Model is learning!")
    print(f"✓ Loss: {final_loss:.4f} (getting close)")
    print(f"✓ Accuracy: {100*correct/15:.0f}% (improving)")
    print("\n📈 NEXT STEPS:")
    print("1. Train for more epochs (150-200)")
    print("2. Or increase dataset to 200-300 examples")
    print("3. Should reach >90% accuracy with more training")
else:
    print("⚠️  Still learning - needs more training")
    print(f"Current loss: {final_loss:.4f} (target: <0.05)")
    print(f"Current accuracy: {100*correct/15:.0f}% (target: >80%)")
    print("\n📈 RECOMMENDATIONS:")
    print("1. Increase epochs to 200")
    print("2. Check if loss is still decreasing")
    print("3. May need more data or longer training")

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
}, "maze_solver_resnet_gpt2.pth")
tokenizer.save_pretrained("maze-model-resnet/final")
print("\n💾 Model saved to maze_solver_resnet_gpt2.pth")
print("💾 Tokenizer saved to maze-model-resnet/final")

CREATING TOKENIZER
Vocabulary: {'R': 4, '<s>': 1, '</s>': 2, '<unk>': 3, 'U': 5, '<pad>': 0}
Vocab size: 6

CREATING MODEL (ResNet18 + GPT2)
Total trainable parameters: 68,680,512
Total trainable parameters: 68,680,512
  - Encoder (ResNet18): 11,571,264
  - Decoder (GPT2-small): 57,110,016

DATASET
Total examples: 924
Training on: 100 examples
(Will scale to full 924 if this works)

TRAINING CONFIGURATION
Device: cuda
Training examples: 100
Batch size: 8
Epochs: 100
Learning rate: 5e-4

STARTING TRAINING


Epoch 1/100: 100%|██████████| 13/13 [00:01<00:00,  8.97it/s, loss=0.9575]


Epoch 1/100, Average Loss: 2.024223


Epoch 2/100: 100%|██████████| 13/13 [00:00<00:00, 14.10it/s, loss=0.6617]


Epoch 2/100, Average Loss: 0.844866


...

Epoch 98/100, Average Loss: 0.444941


Epoch 99/100: 100%|██████████| 13/13 [00:00<00:00, 13.37it/s, loss=0.4709]


Epoch 99/100, Average Loss: 0.446050


Epoch 100/100: 100%|██████████| 13/13 [00:00<00:00, 13.28it/s, loss=0.4371]


Epoch 100/100, Average Loss: 0.457459

Sample Predictions:

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 3:
  Expected:  'R R R R R U U U R U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 4:
  Expected:  'R R R R R U U U U R U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Accuracy: 0/5


TRAINING COMPLETE
Final loss: 0.457459

FINAL TESTING

Sample Predictions:

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Maze 3:
  Expected:  'R R R R R U U U R U U U'
  Predicted: 'R R U U U U U U U U'
  Match: ✗

Ma

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json
from tqdm import tqdm
import torchvision.models as torch_models
from torchvision import transforms

# ============ 1. Tokenizer (Simple mapping) ============
print("=" * 60)
print("SETUP")
print("=" * 60)

token_to_id = {'<pad>': 0, '<s>': 1, '</s>': 2, 'R': 3, 'U': 4}
id_to_token = {v: k for k, v in token_to_id.items()}
vocab_size = len(token_to_id)

print(f"Vocabulary: {token_to_id}")
print(f"Vocab size: {vocab_size}\n")

# ============ 2. Simple Model - ResNet + LSTM ============
class MazeSolver(nn.Module):
    """Direct ResNet18 -> LSTM architecture"""
    def __init__(self, vocab_size=5, hidden_size=256):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        
        # ResNet18 encoder (pretrained)
        resnet = torch_models.resnet18(pretrained=True)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC layer
        
        # Project ResNet features to LSTM hidden size
        self.feature_proj = nn.Linear(512, hidden_size)
        
        # LSTM decoder
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=2, batch_first=True, dropout=0.1)
        self.output_proj = nn.Linear(hidden_size, vocab_size)
        
    def encode_image(self, images):
        """Encode images to feature vectors"""
        features = self.encoder(images)  # [batch, 512, 1, 1]
        features = features.squeeze(-1).squeeze(-1)  # [batch, 512]
        features = self.feature_proj(features)  # [batch, hidden_size]
        return features
    
    def forward(self, images, target_sequences):
        """Training forward pass with teacher forcing"""
        batch_size = images.shape[0]
        
        # Encode images
        img_features = self.encode_image(images)  # [batch, hidden_size]
        
        # Initialize LSTM hidden state with image features
        h0 = img_features.unsqueeze(0).repeat(2, 1, 1)  # [2, batch, hidden_size]
        c0 = torch.zeros_like(h0)
        
        # Prepare input (all tokens except last) and target (all tokens except first)
        input_seq = target_sequences[:, :-1]  # [batch, seq_len-1]
        target_seq = target_sequences[:, 1:]  # [batch, seq_len-1]
        
        # Embed and run through LSTM
        embedded = self.embedding(input_seq)  # [batch, seq_len-1, hidden_size]
        lstm_out, _ = self.lstm(embedded, (h0, c0))  # [batch, seq_len-1, hidden_size]
        
        # Project to vocabulary
        logits = self.output_proj(lstm_out)  # [batch, seq_len-1, vocab_size]
        
        # Compute loss
        loss = nn.CrossEntropyLoss(ignore_index=0)(
            logits.reshape(-1, self.vocab_size),
            target_seq.reshape(-1)
        )
        
        return loss, logits
    
    def generate(self, images, max_length=15, start_token=1, end_token=2):
        """Generate sequences autoregressively"""
        self.eval()
        batch_size = images.shape[0]
        device = images.device
        
        # Encode images
        img_features = self.encode_image(images)
        
        # Initialize LSTM state
        h = img_features.unsqueeze(0).repeat(2, 1, 1)
        c = torch.zeros_like(h)
        
        # Start with <s> token
        current_token = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)
        generated = [current_token]
        
        for _ in range(max_length):
            embedded = self.embedding(current_token)
            lstm_out, (h, c) = self.lstm(embedded, (h, c))
            logits = self.output_proj(lstm_out[:, -1:, :])
            next_token = logits.argmax(dim=-1)
            
            generated.append(next_token)
            current_token = next_token
            
            # Stop if all sequences generated EOS
            if (next_token == end_token).all():
                break
        
        return torch.cat(generated, dim=1)

# ============ 3. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, transform):
        self.entries = entries
        self.transform = transform
    
    def __len__(self):
        return len(self.entries)
    
    def __getitem__(self, idx):
        entry = self.entries[idx]
        
        # Load and transform image
        img = Image.open(entry["image"]).convert("RGB")
        img_tensor = self.transform(img)
        
        # Convert sequence to token IDs: <s> R R R ... </s>
        seq_ids = [token_to_id['<s>']]
        seq_ids.extend([token_to_id[t] for t in entry["sequence"]])
        seq_ids.append(token_to_id['</s>'])
        
        return img_tensor, torch.tensor(seq_ids, dtype=torch.long), entry["sequence"]

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    sequences = [item[1] for item in batch]
    originals = [item[2] for item in batch]
    
    # Pad sequences
    max_len = max(len(s) for s in sequences)
    padded_seqs = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, seq in enumerate(sequences):
        padded_seqs[i, :len(seq)] = seq
    
    return images, padded_seqs, originals

# ============ 4. Training function ============
def train_model(model, train_loader, epochs, lr, device):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, sequences, _ in progress_bar:
            images = images.to(device)
            sequences = sequences.to(device)
            
            # Forward pass
            loss, _ = model(images, sequences)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.6f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
        
        # Test every 20 epochs
        if (epoch + 1) % 20 == 0:
            test_model(model, train_loader, device, num_samples=5)
            model.train()
    
    return best_loss

def test_model(model, loader, device, num_samples=10):
    model.eval()
    dataset = loader.dataset
    
    print("\n" + "="*60)
    print("Sample Predictions:")
    print("="*60)
    
    correct = 0
    for i in range(min(num_samples, len(dataset))):
        img_tensor, _, original_seq = dataset[i]
        img_tensor = img_tensor.unsqueeze(0).to(device)
        
        with torch.no_grad():
            generated = model.generate(img_tensor, max_length=20)
        
        # Decode prediction
        pred_tokens = []
        for token_id in generated[0].cpu().tolist()[1:]:  # Skip <s>
            if token_id == token_to_id['</s>']:
                break
            if token_id in [3, 4]:  # R or U
                pred_tokens.append(id_to_token[token_id])
        
        expected = original_seq
        match = pred_tokens == expected
        if match:
            correct += 1
        
        print(f"\nMaze {i}:")
        print(f"  Expected:  {' '.join(expected)}")
        print(f"  Predicted: {' '.join(pred_tokens)}")
        print(f"  Match: {'✓' if match else '✗'}")
    
    print(f"\nAccuracy: {correct}/{num_samples}\n")
    return correct

# ============ 5. Main training ============
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load data
entries = json.load(open("data/grid_sequences.json"))
print(f"Total examples: {len(entries)}")

# Use 100 examples for testing
train_entries = entries[:100]
print(f"Training on: {len(train_entries)} examples\n")

# Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create dataset and dataloader
dataset = MazeDataset(train_entries, transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

# Create model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = MazeSolver(vocab_size=vocab_size, hidden_size=256)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}\n")

# Train
print("=" * 60)
print("TRAINING")
print("=" * 60)
print("This simpler architecture should converge!\n")

final_loss = train_model(model, train_loader, epochs=100, lr=1e-3, device=device)

# Final test
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

correct = test_model(model, train_loader, device, num_samples=20)

print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Final Loss: {final_loss:.6f}")
print(f"Accuracy: {correct}/20 ({100*correct/20:.0f}%)")
print("=" * 60)

if final_loss < 0.1 and correct >= 16:
    print("\n🎉 SUCCESS! Model works!")
    print("✅ Ready to scale to full dataset")
elif final_loss < 0.5 and correct >= 5:
    print("\n⚠️ Partial success - needs more training")
    print("Try: 150-200 epochs or more data")
else:
    print("\n❌ Still not learning well")
    print("May need to debug further")

# Save
torch.save(model.state_dict(), "maze_solver_simple.pth")
print("\n💾 Model saved to maze_solver_simple.pth")

SETUP
Vocabulary: {'<pad>': 0, '<s>': 1, '</s>': 2, 'R': 3, 'U': 4}
Vocab size: 5

LOADING DATA
Total examples: 924
Training on: 100 examples

Device: cuda
Total parameters: 12,363,077
Trainable parameters: 12,363,077

TRAINING
This simpler architecture should converge!



Epoch 1/100: 100%|██████████| 7/7 [00:00<00:00,  7.78it/s, loss=0.7422]


Epoch 1, Avg Loss: 1.076394


Epoch 2/100: 100%|██████████| 7/7 [00:00<00:00, 16.32it/s, loss=0.5564]


Epoch 2, Avg Loss: 0.660294


...

Epoch 98, Avg Loss: 0.299205


Epoch 99/100: 100%|██████████| 7/7 [00:00<00:00, 15.04it/s, loss=0.6052]


Epoch 99, Avg Loss: 0.293340


Epoch 100/100: 100%|██████████| 7/7 [00:00<00:00, 14.97it/s, loss=0.2913]


Epoch 100, Avg Loss: 0.238497

Sample Predictions:

Maze 0:
  Expected:  R R R R R R U U U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✗

Maze 1:
  Expected:  R R R R R U R U U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✗

Maze 2:
  Expected:  R R R R R U U R U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✓

Maze 3:
  Expected:  R R R R R U U U R U U U
  Predicted: R R R R R U U U U R U U
  Match: ✗

Maze 4:
  Expected:  R R R R R U U U U R U U
  Predicted: R R R R U U R U U R U U
  Match: ✗

Accuracy: 1/5


FINAL EVALUATION

Sample Predictions:

Maze 0:
  Expected:  R R R R R R U U U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✗

Maze 1:
  Expected:  R R R R R U R U U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✗

Maze 2:
  Expected:  R R R R R U U R U U U U
  Predicted: R R R R R U U R U U U U
  Match: ✓

Maze 3:
  Expected:  R R R R R U U U R U U U
  Predicted: R R R R R U U U U R U U
  Match: ✗

Maze 4:
  Expected:  R R R R R U U U U R U U
  

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json
from tqdm import tqdm
import torchvision.models as torch_models
from torchvision import transforms
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import PreTrainedTokenizerFast

# ============ 1. Create tokenizer ============
print("=" * 60)
print("TOKENIZER SETUP")
print("=" * 60)

tok = Tokenizer(models.WordLevel(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.WordLevelTrainer(special_tokens=["<pad>", "<s>", "</s>", "<unk>"])
tok.train_from_iterator(["R U R U", "U R U R"], trainer)
tok.save("maze_tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="maze_tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>"
)

print(f"Vocabulary: {tokenizer.get_vocab()}")
print(f"Vocab size: {len(tokenizer)}\n")

# ============ 2. ResNet + GPT2 Model with Proper Cross-Attention ============
class ResNetGPT2Solver(nn.Module):
    """ResNet18 encoder + GPT2 decoder with verified cross-attention"""
    def __init__(self, vocab_size, gpt2_hidden_size=768):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = gpt2_hidden_size
        
        # ResNet18 encoder (pretrained)
        resnet = torch_models.resnet18(pretrained=True)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-1])
        
        # Project ResNet features to GPT2 hidden size and expand to sequence
        self.feature_projection = nn.Linear(512, gpt2_hidden_size)
        
        # Create multiple "image tokens" from single feature vector
        self.image_token_expansion = nn.Linear(gpt2_hidden_size, gpt2_hidden_size * 8)
        
        # GPT2 decoder with cross-attention enabled
        gpt2_config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=64,
            n_embd=gpt2_hidden_size,
            n_layer=4,  # Smaller for faster training
            n_head=12,
            add_cross_attention=True,  # CRITICAL: Enable cross-attention
            use_cache=False,
        )
        self.gpt2 = GPT2LMHeadModel(gpt2_config)
        
    def encode_image(self, images):
        """Encode images to feature sequence for cross-attention"""
        batch_size = images.shape[0]
        
        # Extract ResNet features
        features = self.resnet_features(images)  # [batch, 512, 1, 1]
        features = features.squeeze(-1).squeeze(-1)  # [batch, 512]
        
        # Project to GPT2 dimension
        features = self.feature_projection(features)  # [batch, hidden_size]
        
        # Expand to multiple tokens (8 tokens per image)
        expanded = self.image_token_expansion(features)  # [batch, hidden_size * 8]
        expanded = expanded.view(batch_size, 8, self.hidden_size)  # [batch, 8, hidden_size]
        
        return expanded
    
    def forward(self, images, input_ids, labels=None):
        """Forward pass with cross-attention to image features"""
        # Encode images
        encoder_hidden_states = self.encode_image(images)  # [batch, 8, hidden_size]
        
        # Forward through GPT2 with cross-attention
        outputs = self.gpt2(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,  # This enables cross-attention!
            labels=labels,
            return_dict=True
        )
        
        return outputs
    
    def generate(self, images, max_length=20, bos_token_id=1, eos_token_id=2, pad_token_id=0):
        """Generate sequences with cross-attention to images"""
        self.eval()
        batch_size = images.shape[0]
        device = images.device
        
        # Encode images
        encoder_hidden_states = self.encode_image(images)
        
        # Start with BOS token
        generated = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
        
        for _ in range(max_length - 1):
            # Forward with cross-attention
            outputs = self.gpt2(
                input_ids=generated,
                encoder_hidden_states=encoder_hidden_states,
                return_dict=True
            )
            
            # Get next token
            next_token_logits = outputs.logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)
            
            # Append
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop if all sequences end
            if (next_token == eos_token_id).all():
                break
        
        return generated

# ============ 3. Dataset ============
class MazeDataset(Dataset):
    def __init__(self, entries, transform, tokenizer):
        self.entries = entries
        self.transform = transform
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.entries)
    
    def __getitem__(self, idx):
        entry = self.entries[idx]
        
        # Load image
        img = Image.open(entry["image"]).convert("RGB")
        img_tensor = self.transform(img)
        
        # Tokenize sequence: <s> R R R ... </s>
        sequence_text = " ".join(entry["sequence"])
        tokens = self.tokenizer(sequence_text, return_tensors="pt", add_special_tokens=True)
        input_ids = tokens.input_ids[0]
        
        return img_tensor, input_ids, entry["sequence"]

def collate_fn(batch, pad_token_id=0):
    images = torch.stack([item[0] for item in batch])
    sequences = [item[1] for item in batch]
    originals = [item[2] for item in batch]
    
    # Pad sequences
    max_len = max(len(s) for s in sequences)
    padded = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    
    return images, padded, originals

# ============ 4. Training ============
def train_model(model, train_loader, epochs, lr, device):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, sequences, _ in progress_bar:
            images = images.to(device)
            sequences = sequences.to(device)
            
            # Prepare input (without last token) and labels (without first token)
            input_ids = sequences[:, :-1]
            labels = sequences[:, 1:].contiguous()
            
            # Replace padding in labels with -100 (ignored in loss)
            labels[labels == 0] = -100
            
            # Forward
            outputs = model(images, input_ids, labels)
            loss = outputs.loss
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.6f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
        
        # Test every 25 epochs
        if (epoch + 1) % 25 == 0:
            test_model(model, train_loader, device, tokenizer, num_samples=5)
            model.train()
    
    return best_loss

def test_model(model, loader, device, tokenizer, num_samples=10):
    model.eval()
    dataset = loader.dataset
    
    print("\n" + "="*60)
    print("Sample Predictions:")
    print("="*60)
    
    correct = 0
    for i in range(min(num_samples, len(dataset))):
        img_tensor, _, original_seq = dataset[i]
        img_tensor = img_tensor.unsqueeze(0).to(device)
        
        with torch.no_grad():
            generated = model.generate(
                img_tensor,
                max_length=20,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        
        # Decode
        predicted = tokenizer.decode(generated[0], skip_special_tokens=True)
        expected = " ".join(original_seq)
        
        match = predicted == expected
        if match:
            correct += 1
        
        print(f"\nMaze {i}:")
        print(f"  Expected:  '{expected}'")
        print(f"  Predicted: '{predicted}'")
        print(f"  Match: {'✓' if match else '✗'}")
    
    print(f"\nAccuracy: {correct}/{num_samples}\n")
    return correct

# ============ 5. Main ============
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

entries = json.load(open("data/grid_sequences.json"))
print(f"Total examples: {len(entries)}")

train_entries = entries[:100]
print(f"Training on: {len(train_entries)} examples\n")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = MazeDataset(train_entries, transform, tokenizer)
train_loader = DataLoader(
    dataset, 
    batch_size=8, 
    shuffle=True, 
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = ResNetGPT2Solver(vocab_size=len(tokenizer), gpt2_hidden_size=768)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

print("=" * 60)
print("TRAINING - ResNet18 Encoder + GPT2 Decoder")
print("=" * 60)
print("Cross-attention is properly enabled!\n")

final_loss = train_model(model, train_loader, epochs=150, lr=5e-4, device=device)

print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

correct = test_model(model, train_loader, device, tokenizer, num_samples=15)

print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Final Loss: {final_loss:.6f}")
print(f"Accuracy: {correct}/15 ({100*correct/15:.1f}%)")
print("=" * 60)

if final_loss < 0.1 and correct >= 12:
    print("\n🎉 SUCCESS! ResNet + GPT2 works!")
    print("✅ Image encoder (ResNet) → LLM decoder (GPT2)")
    print("✅ Ready to scale to full dataset")
elif final_loss < 0.3 and correct >= 5:
    print("\n⚠️ Getting better! Try more epochs (200-250)")
else:
    print("\n❌ Still struggling")
    print("May need different approach or more investigation")

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
}, "resnet_gpt2_solver.pth")
print("\n💾 Model saved!")

TOKENIZER SETUP
Vocabulary: {'<pad>': 0, '<s>': 1, '<unk>': 3, 'R': 4, 'U': 5, '</s>': 2}
Vocab size: 6

LOADING DATA
Total examples: 924
Training on: 100 examples

Device: cuda
Parameters: 54,157,632

TRAINING - ResNet18 Encoder + GPT2 Decoder
Cross-attention is properly enabled!



Epoch 1/150: 100%|██████████| 13/13 [00:01<00:00, 11.99it/s, loss=0.7641]


Epoch 1, Avg Loss: 2.038752


Epoch 2/150: 100%|██████████| 13/13 [00:00<00:00, 15.02it/s, loss=0.6827]


Epoch 2, Avg Loss: 0.752070


...

Epoch 148, Avg Loss: 0.590075


Epoch 149/150: 100%|██████████| 13/13 [00:00<00:00, 14.75it/s, loss=0.6746]


Epoch 149, Avg Loss: 0.591404


Epoch 150/150: 100%|██████████| 13/13 [00:00<00:00, 14.88it/s, loss=0.5275]


Epoch 150, Avg Loss: 0.594743

Sample Predictions:

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R U U U R U U U U U U U U U U U U U'
  Match: ✗

Maze 3:
  Expected:  'R R R R R U U U R U U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Maze 4:
  Expected:  'R R R R R U U U U R U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Accuracy: 0/5


FINAL EVALUATION

Sample Predictions:

Maze 0:
  Expected:  'R R R R R R U U U U U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Maze 1:
  Expected:  'R R R R R U R U U U U U'
  Predicted: 'R R U R U U R U U U U U U U U U U U U'
  Match: ✗

Maze 2:
  Expected:  'R R R R R U U R U U U U'
  Predicted: 'R R U U U R U U U U U U U U U U U U U'
  Match: 